# Лабораторная работа 5. Перцептрон Розенблатта

**Вариант 2. Линейный тренд с шумом Лапласа, окно из шести наблюдений.**

Предскажем следующее значение ряда по шести предыдущим. По форме предсказания это линейная регрессия из лабы 3 с несколькими признаками. Веса найдём градиентным спуском из лабы 2.

Запускай ячейки сверху вниз. Строки с `TODO` дописывай вместе с учеником, затем запускай проверки.

[Условие задания](lab5.md) · [Урок с выводом формул](lesson/lab5.md) · [ГДЗ](lesson/lab5_gdz.ipynb)

## 1. Генерация сигнала

Берём 600 равноотстоящих моментов $t=0,\ldots,599$. Чистый тренд задаётся формулой $f(t)=0.03t-5$. К каждой точке прибавляем независимый шум Лапласа с центром $0$ и параметром масштаба $0.3$. Получаем наблюдаемый ряд $y_t=f(t)+\varepsilon_t$.

Как в лабе 1, используем отдельный генератор случайных чисел. Фиксированный seed позволяет сравнивать эксперименты на одном и том же ряде.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 600
WINDOW = 6
SPLIT_AT = int(0.8 * N)
DATA_SEED = 42

rng = np.random.default_rng(DATA_SEED)
t = np.arange(N)
clean_signal = 0.03 * t - 5
noise = rng.laplace(loc=0.0, scale=0.3, size=N)
signal = clean_signal + noise

plt.figure(figsize=(10, 3.5))
plt.scatter(t, signal, s=9, alpha=0.45, color="tab:blue", label="Наблюдения")
plt.plot(t, clean_signal, color="tab:orange", linewidth=2, label="Чистый тренд")
plt.xlabel("Время t")
plt.ylabel("Значение ряда")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 2. Окна и разделение по времени

Для прогноза $y_t$ берём строку из значений $y_{t-6},\ldots,y_{t-1}$. Один ряд превращается в таблицу. Строка содержит один пример, столбец соответствует одной позиции в окне, цель хранится отдельно.

Разрежем исходный ряд по времени в точке $t=480$. Цели с $t<480$ идут в train. В test идут цели с $t\geq480$. Первый тестовый прогноз использует $y_{474},\ldots,y_{479}$. Затем прогнозируем на один шаг вперёд, добавляя фактическое наблюдение в историю после того, как оно стало доступно.

Спроси, какие значения будут входами и целью для первого окна примера.

In [ ]:
def make_windows(series, size):
    # TODO: собери строки [end - size : end] для end от size до конца ряда.
    # Цель строки имеет индекс end. Верни матрицу X и вектор y.
    ...


X_demo, y_demo = make_windows(np.array([1, 2, 3, 4, 5]), 2)
np.testing.assert_array_equal(X_demo, [[1, 2], [2, 3], [3, 4]])
np.testing.assert_array_equal(y_demo, [3, 4, 5])
print(X_demo, y_demo, sep="\n")

In [ ]:
X, y = make_windows(signal, WINDOW)
target_t = t[WINDOW:]

# TODO: сделай маску train по времени цели, test получи обратной маской ~.
# Сохрани X_train, X_test, y_train, y_test, t_train, t_test.

assert X_train.shape == (474, WINDOW) and X_test.shape == (120, WINDOW)
assert t_train[-1] == 479 and t_test[0] == 480
np.testing.assert_array_equal(X_test[0], signal[SPLIT_AT - WINDOW:SPLIT_AT])
print("Первое тестовое окно:", X_test[0].round(3))

## 3. Нормализация входов

Как в лабе 4, стандартизируем каждый столбец по формуле $z_j=(x_j-\mu_j)/\sigma_j$. Средние и стандартные отклонения вычисляем по train. К test применяем те же числа. Цель $y$ оставляем в единицах исходного ряда.

In [ ]:
# TODO: найди center и scale по столбцам X_train (axis=0).
# Получи Z_train и Z_test с этими же center и scale.

np.testing.assert_allclose(Z_train.mean(axis=0), 0.0, atol=1e-12)
np.testing.assert_allclose(Z_train.std(axis=0), 1.0)

## 4. Линейный нейрон и метрики

Нейрон складывает взвешенные входы и добавляет смещение $b$. Линейная активация здесь просто возвращает полученное число.

$$
\hat y = w_1z_1 + \cdots + w_6z_6 + b.
$$

MSE и MAPE уже были в лабе 3. MSE усредняет квадраты ошибок. MAPE усредняет абсолютные относительные ошибки в процентах. В реализации ниже при точном $y_i=0$ слагаемое MAPE пропускается, а при отсутствии ненулевых целей функция возвращает `nan`. При малом ненулевом $|y_i|$ вклад точки остаётся большим.

In [ ]:
def predict(Z, w, b):
    # TODO: матричное умножение даёт одно число на строку.
    ...


def mse(y, y_pred):
    # TODO: возведи ошибки в квадрат, затем возьми среднее.
    ...


def mape(y, y_pred):
    # TODO: маска для ненулевых целей, среднее абсолютных отношений, перевод в %.
    # Если ненулевых целей нет, верни np.nan.
    ...



# Первые две строки учебного примера после стандартизации.
Z_demo = np.array([[-1.0, -1.0], [1.0, 1.0]])
y_demo_train = y_demo[:2]
w_demo = np.array([0.5, -0.5])
b_demo = 3.0
pred_demo = predict(Z_demo, w_demo, b_demo)

np.testing.assert_allclose(pred_demo, [3.0, 3.0])
np.testing.assert_allclose(mse(y_demo_train, pred_demo), 0.5)
np.testing.assert_allclose(mape(y_demo_train, pred_demo), 12.5)

## 5. Дельта-правило и обучение

Как в лабе 2, один шаг идёт по антиградиенту. Здесь потери — MSE, параметры — вектор весов и смещение. Считаем вектор ошибок $e=y-\hat y$ на всех $n$ строках train. Для одного обновления используем одну и ту же ошибку в обеих формулах.

$$
w\gets w + \eta \frac{2}{n} Z^\top e,\qquad
b\gets b + 2\eta\,\operatorname{mean}(e).
$$

В этой реализации один полный проход по train даёт одно обновление. Скорость обучения $\eta$ постоянна внутри запуска. Сохраним параметры до первого шага и после каждой эпохи, чтобы позднее посчитать метрики.

Попроси предсказать, в какую сторону изменится вес в числовом примере.

In [ ]:
def delta_step(Z, y, w, b, lr):
    # TODO: вычисли e = y - y_pred. Используй этот же e для новых w и b.
    ...



w_after, b_after = delta_step(Z_demo, y_demo_train, w_demo, b_demo, lr=0.25)
np.testing.assert_allclose(w_after, [0.75, -0.25])
np.testing.assert_allclose(b_after, 3.25)

# Первое тестовое окно [3, 4] в масштабе обучающих столбцов превращается в [3, 3].
pred_next = predict(np.array([[3.0, 3.0]]), w_after, b_after)
np.testing.assert_allclose(pred_next, [4.75])

In [ ]:
def fit(Z_train, y_train, lr, epochs, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.1, size=Z_train.shape[1])
    b = rng.normal(0, 0.1)
    trajectory = [(w, b)]

    # TODO: за каждую эпоху сделай один delta_step, добавь (w, b) в trajectory.

    return w, b, trajectory

## 6. Графики и проверка по МНК

Посчитаем train/test MSE и MAPE для каждой сохранённой пары параметров. `fit` получает только обучающие данные. Test используем для просмотра качества одношагового прогноза.

In [ ]:
def evaluate_history(trajectory, Z_train, y_train, Z_test, y_test):
    history = {"train_mse": [], "test_mse": [], "train_mape": [], "test_mape": []}
    # TODO: для каждой пары (w, b) получи два набора прогнозов и заполни четыре списка.
    return history


def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, metric, label in zip(axes, ["mse", "mape"], ["MSE", "MAPE, %"]):
        ax.plot(history[f"train_{metric}"], label="Train")
        ax.plot(history[f"test_{metric}"], label="Test")
        ax.set(xlabel="Эпоха", ylabel=label, yscale="log")
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    plt.show()


LR = 0.1
EPOCHS = 200
w, b, trajectory = fit(Z_train, y_train, LR, EPOCHS)
history = evaluate_history(trajectory, Z_train, y_train, Z_test, y_test)
assert len(trajectory) == EPOCHS + 1
assert len(history["train_mse"]) == len(trajectory)
plot_history(history)

Сопоставим прогноз с наблюдаемым рядом и чистым трендом. Чистый тренд известен благодаря генерации данных. Метрики считаем относительно наблюдаемого ряда с шумом.

In [ ]:
pred_train = predict(Z_train, w, b)
pred_test = predict(Z_test, w, b)

# TODO: на одном графике покажи signal, clean_signal, pred_train и pred_test.
# Для предсказаний используй t_train и t_test. Отметь границу SPLIT_AT.
# Рассмотри крупнее первые 40 тестовых точек.

Вернёмся к лабе 3. МНК и градиентный спуск минимизируют одну и ту же сумму квадратов с точностью до постоянного делителя $n$. Добавим к входам столбец единиц для смещения и найдём контрольное решение через `np.linalg.lstsq`. Для сравнения возьмём простой прогноз по последнему наблюдению окна.

In [ ]:
A_train = np.column_stack((Z_train, np.ones(len(Z_train))))

# TODO: найди коэффициенты через np.linalg.lstsq(A_train, y_train, rcond=None).
# Последний коэффициент — смещение. Получи прогнозы и посчитай MSE/MAPE.
# Сравни с прогнозом X_test[:, -1] и результатом градиентного спуска.
# Выведи минимальное |y| отдельно для train и test.

Сравни кривые MSE и MAPE. Почему train MAPE может быть намного выше test MAPE, хотя train MSE меньше test MSE? Какую картину показывает увеличенный тестовый отрезок?

## 7. Эксперименты

### Скорость обучения

Сравним $\eta=0.001$, $0.1$ и $0.17$ при 60 эпохах. Зафиксируем начальные веса и данные. Логарифмическая шкала MSE позволяет увидеть все три кривые на одном рисунке.

In [ ]:
rates = (0.001, 0.1, 0.17)
rate_epochs = 60

# TODO: для каждого rate запусти fit с одинаковым числом эпох и начальным seed.
# Вызови evaluate_history. Сравни train MSE и test MSE на двух графиках.
# Используй логарифмическую шкалу по оси Y, например ax.set_yscale("log").

При каком шаге ошибка ещё снижается, при каком вышла на плато и при каком растёт? Сравни с размером шага в лабе 2.

### Число эпох

Теперь фиксируем шаг $\eta=0.1$ и сравниваем 5, 40 и 200 эпох. Данные и начальные параметры снова одни и те же.

In [ ]:
epoch_counts = (5, 40, 200)

# TODO: для каждого count вызови fit с одним LR, но своим числом эпох.
# Сравни итоговые train MSE и test MSE в таблице или на графике.

Сильно ли меняются метрики между 40 и 200 эпохами? Если бы нужно было выбрать число эпох для рабочей задачи, какой отрезок ряда можно было бы выделить для валидации?

## 8. Вопросы для защиты

1. Что такое переобучение и как его избежать в задаче регрессии?
2. Почему важно нормализовать входы перед обучением?
3. Как связана скорость обучения со сходимостью?
4. Может ли один линейный нейрон аппроксимировать любую функцию?
5. Как по графикам заметить недообучение или переобучение?